In [1]:
using LinearAlgebra, Plots, Distributions, LaTeXStrings, Random, DataFrames, CSV

In [ ]:
include(joinpath(@__DIR__, "..", "..", "AuxiliaryFunctions.jl"))
include(joinpath(@__DIR__, "..", "..", "SpikeEstimation.jl"))
figdir = "Figures"
tbdir = "Tables"

LegQuadInt (generic function with 1 method)

In [ ]:
Random.seed!(1234)

N = 6000
d = 0.1
M = convert(Int64,ceil(N/d))
X = randn(N,M)

K = 200
nodes, weights = LegQuad(K)
a = 0.1; b = 4.0
h = x-> a<x<b ? (x^4+1)*sqrt(x-a)*sqrt(b-x)/x^2 : 0
normCst = LegQuadInt(h,a,b,nodes,weights)
scaled_h = x->h(x)/normCst
quantiles = zeros(Float64,N+1)
quantiles[1] = a
quantiles[N+1] = b
for i=2:N
    QuantEq = x->LegQuadInt(scaled_h,a,x,nodes,weights)-(i-1)/N
    quantiles[i] = Bisection(QuantEq,quantiles[1],quantiles[N+1])
end

δ = 6
quantiles[1:3] = [7,δ,δ]
sqrtΣ = Diagonal(sqrt.(quantiles[1:end-1]))
W = Matrix{Float64}(undef,N,N)
rmul!(X, inv(sqrt(M)))
mul!(X,sqrtΣ,X)
mul!(W,X,X')
evals = eigvals(Symmetric(W))

true_spikes = evals[end:-1:end-2]
p1 = histogram(quantiles,bins=quantiles[4]-0.2:0.1:quantiles[1]+0.2,normalize=:pdf,label="ESD of Σ",framestyle=:box, legendfontsize=12, xtickfontsize=12, ytickfontsize=12)
color = theme_palette(:auto).colors[1]
p1 = scatter!(quantiles[1:3],0*quantiles[1:3],markersize=8,color=color,marker=:dot,label="Spikes of Σ",alpha=0.5)

results = AsympSCM(W, vecNbr=150, nx=2000, tol=2.0/sqrt(N))
xvec, yvec, SpikeNbr, SpikeLoc, γmin, γplus = results[:x], results[:density], results[:spikes_nbr], results[:spikes_loc], results[:γmin], results[:γplus]

p2 = histogram(evals,bins=γmin-0.2:0.1:γplus+0.2,normalize=:pdf,label="ESD of "*L"W",framestyle=:box, legendfontsize=12, xtickfontsize=12, ytickfontsize=12)
color = theme_palette(:auto).colors[1]
p2 = plot!(xvec,yvec,linecolor=:red,linewidth=4,label="Estimated density")
p2 = scatter!(true_spikes,0*true_spikes,markersize=8,color=color,marker=:dot,label="True Outliers")
p2 = scatter!(SpikeLoc,0*SpikeLoc,markersize=5,color=:red,marker=:dot,label="Estimated Outliers")
savefig(p1,joinpath(figdir, "Sigma.pdf"))
savefig(p2,joinpath(figdir, "Density.pdf"))
tb = DataFrame(A=true_spikes,B=SpikeLoc,C=abs.(true_spikes-SpikeLoc))
CSV.write(joinpath(tbdir, "Spikes.csv"),tb)

"DemoSpikes.csv"